<a href="https://colab.research.google.com/github/QuyDuong106/INFOSYS722_BDAS/blob/main/run_spark_lopo_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Spark RFE + LOPO: cached preparation and CUDA tuning
This notebook loads the existing project from Google Drive. It does not write bundled source or copy an embedded dataset.

Project path: `/content/drive/MyDrive/INFOSYS_722_Notebook/iteration4_BDAS`.
Put the updated `spark_lopo.py` and `spark_lopo_acceleration.py` alongside the notebook in that folder. The folder must also contain the existing `spark_workflow.py`, `spark_iterations.py`, `spark_feature_selection.py` and `data/raw/dataset_gallstone`.

For each of five seeds, each patient record is held out once. The three native candidates (logistic regression, random forest, GBT), plus the optional CUDA XGBoost candidate, train on the remaining records. Native model grids and XGBoost Optuna trials use stratified CV inside the outer training data. Prepared training/validation partitions are cached once and shared across all candidates. RFE and imputation/scaling are fitted inside every training fold. Only the CV-selected candidate configuration predicts the held-out patient; held-out labels never select parameters.

Accuracy, precision and recall are first calculated from all pooled held-out predictions for each seed. The report shows their mean and sample standard deviation across the five complete runs. Precision and recall from single-patient folds are not averaged because those folds often have undefined denominators. Seeds change training randomness/CV allocation; they reuse the same patients, so the SD describes algorithm variability.


In [2]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
PROJECT_ROOT = Path('/content/drive/MyDrive/INFOSYS_722_Notebook/iteration4_BDAS')
DATA_PATH = PROJECT_ROOT/'data/raw/dataset_gallstone'
OUTPUT_ROOT = PROJECT_ROOT/'results/spark_lopo_rfe'
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f'Project folder not found: {PROJECT_ROOT}')


Mounted at /content/drive


In [ ]:
import sys, subprocess, importlib.util, os, shutil
if importlib.util.find_spec('pyspark') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pyspark>=3.5.3,<4.0'])
if importlib.util.find_spec('sklearn') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'scikit-learn'])
for module, requirement in [('xgboost', 'xgboost>=2.1,<3.1'), ('optuna', 'optuna>=4,<6')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', requirement])
JAVA17 = Path('/usr/lib/jvm/java-17-openjdk-amd64')
if not JAVA17.exists() and not shutil.which('java'):
    subprocess.check_call(['apt-get', 'update', '-qq'])
    subprocess.check_call(['apt-get', 'install', '-y', '-qq', 'openjdk-17-jre-headless'])
if JAVA17.exists():
    os.environ['JAVA_HOME'] = str(JAVA17)
# Load your Drive scripts, without writing source files.
required = ['spark_lopo.py','spark_lopo_acceleration.py','spark_cv_reporting.py','spark_workflow.py','spark_iterations.py','spark_feature_selection.py']
missing = [name for name in required if not (PROJECT_ROOT/name).is_file()]
if missing:
    raise FileNotFoundError(f'Copy these updated scripts to {PROJECT_ROOT}: {missing}')
if not DATA_PATH.is_file():
    raise FileNotFoundError(DATA_PATH)
sys.path.insert(0, str(PROJECT_ROOT))
for module in ['spark_lopo','spark_lopo_acceleration','spark_cv_reporting','spark_iterations','spark_feature_selection','spark_workflow']:
    sys.modules.pop(module, None)
from spark_workflow import create_spark, load_dataset
from spark_lopo import run_repeated_lopo


## RFE and candidate settings
Default: RFE20. To use ten selected fields, change `VARIANT` to `rfe10_standard`; no CART or all-field option is used by this LOPO workflow.

Select **Runtime → Change runtime type → GPU** before running. The default includes `xgboost_gpu`: Optuna runs its boosted-tree fits with `device="cuda"` and `tree_method="hist"`. CUDA is checked before the LOPO workload; CPU fallback is rejected. Native Spark LR/RF/GBT, imputation and CART-based RFE still run on CPU.

`MODELS` includes all three native Spark candidates plus CUDA XGBoost. Native grids remain LR: three configurations, RF: four, GBT: four. `GPU_TRIALS=20` controls training-only XGBoost Optuna trials. This is explicitly a hybrid workflow: Spark fits preparation and calculates pooled results, while a bounded prepared matrix (maximum 20,000 rows) is collected to the Colab driver for XGBoost. It does not claim Spark ML runs on GPU. GBT and XGBoost remain distinct models.

Every inner fold learns RFE/scaling from its own training subset. Its prepared partitions are reused across models and hyperparameter candidates, never across incompatible patient splits. Fully completed outer folds skip Spark training work. Interrupted GPU studies restore completed trials from Drive JSON. A single GPU runs trials sequentially to avoid contention. GPU overhead may outweigh training savings on this 319-record dataset; no speedup is guaranteed.

A quick partial check uses `MAX_FOLDS=2`. To finish or resume, set it to `None` with the other settings unchanged. Partial checks do not write final metrics or mean/SD summaries.


In [ ]:
VARIANT = 'rfe20_standard'
MODELS = ['logistic', 'random_forest', 'gbt', 'xgboost_gpu']
# For a GPU-only candidate run: MODELS = ['xgboost_gpu']
GPU_TRIALS = 20
USE_GPU = True
SEEDS = [42, 7, 21, 84, 123]
TUNE = True
INNER_FOLDS = 3
MAX_FOLDS = None
spark = create_spark(master='local[2]', app_name='Research-RFE-LOPO')
frame, numeric, categorical = load_dataset(spark, DATA_PATH)
frame = frame.cache()
frame.count()  # Reuse source parsing/validation across all outer folds and seeds.
print('Patients:', frame.count(), '| Predictors:', len(numeric)+len(categorical))
print('Data:', DATA_PATH, '\nOutputs:', OUTPUT_ROOT)
frame.groupBy('label').count().show()


In [ ]:
if 'xgboost_gpu' in MODELS and USE_GPU:
    from spark_lopo_acceleration import check_cuda
    print('Verified XGBoost device:', check_cuda())
else:
    print('Native Spark candidates use CPU; no CUDA tuning is requested.')


## Train, test each held-out patient, and resume checkpoints

In [ ]:
metrics_per_seed, summary, run_directories = run_repeated_lopo(
    spark, frame, numeric, categorical, data_path=DATA_PATH,
    output_dir=OUTPUT_ROOT, seeds=SEEDS, variant=VARIANT, models=MODELS,
    tune=TUNE, inner_folds=INNER_FOLDS, max_folds=MAX_FOLDS, gpu_trials=GPU_TRIALS, use_gpu=USE_GPU)
if summary is not None:
    print('Mean and sample standard deviation across complete LOPO seed runs:')
    summary.orderBy('model').show(truncate=False)
    metrics_per_seed.select('seed','model','accuracy','precision','recall','roc_auc',
        'TN','FP','FN','TP').orderBy('model','seed').show(truncate=False)


## Mean ± standard deviation report

In [ ]:
if summary is not None:
    from pyspark.sql import functions as F
    report = summary.select('model', *[
        F.when(F.col(metric+'_std').isNotNull(),
            F.format_string('%.4f ± %.4f', F.col(metric+'_mean'), F.col(metric+'_std'))
        ).otherwise(F.lit('undefined / fewer than two valid runs')).alias(metric)
        for metric in ['accuracy','precision','recall']])
    report.orderBy('model').show(truncate=False)


## Average LOPO confusion matrix across seeds

In [ ]:
if metrics_per_seed is not None:
    import matplotlib.pyplot as plt
    averages = metrics_per_seed.groupBy('model').agg(*[
        F.mean(c).alias(c) for c in ['TN','FP','FN','TP']]).collect()
    for row in averages:
        matrix = [[row.TN,row.FP],[row.FN,row.TP]]
        fig, ax = plt.subplots(figsize=(4,4))
        ax.imshow(matrix, cmap='Blues')
        for i in range(2):
            for j in range(2):
                ax.text(j,i,f'{matrix[i][j]:.2f}',ha='center',va='center')
        ax.set(xticks=[0,1],yticks=[0,1],xticklabels=['Absent','Present'],
               yticklabels=['Absent','Present'],xlabel='Predicted',ylabel='Actual',
               title=f'{row.model}: average LOPO counts')
        fig.tight_layout()
        # Save next to the summary, in a folder identified by these exact run directories.
        import hashlib, json
        identity = dict(run_directories=[str(p) for p in run_directories], seeds=SEEDS,
            aggregation='mean and sample standard deviation of pooled LOPO metrics across seeds')
        signature = hashlib.sha256(json.dumps(identity,sort_keys=True).encode()).hexdigest()[:12]
        fig.savefig(OUTPUT_ROOT/'summaries'/signature/f'average_cm_{row.model}.png',dpi=200)
        plt.show()


All outputs stay under your project's `results/spark_lopo_rfe` folder. Individual runs retain `cv_search.json`, selected fields, per-patient prediction checkpoints, `predictions.csv` and pooled metrics. `summaries/<configuration>/summary.csv` contains the requested mean and sample SD; `metrics_per_seed.csv` preserves the measurements used to calculate them.

Undefined precision (no predicted positives) is stored as null; the summary includes valid-run counts and excludes nulls rather than silently setting precision to zero. Five seeded LOPO runs with nested RFE/CV require considerably more computation than a single holdout run. This remains separate from the report's holdout protocol.
